In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.glutamate.summary import GlutamateSummary
from vip_slap2_analysis.utils.utils import normalize

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [3]:
savepath = r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures'

In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd


def _image_label_from_key(k):
    return Path(str(k).replace("\\", "/")).stem


def _safe_session_order(metadata):
    """
    Extract session order from keys like 'session_#' if present.
    """
    if metadata is None:
        return np.nan

    if "session_#" in metadata:
        return metadata["session_#"]

    # fallback: try a few common variants
    for key in ["session_num", "session_number", "session_order"]:
        if key in metadata:
            return metadata[key]

    return np.nan


import numpy as np

def _compute_variance_decomposition_for_synapse_amplitude(
    image_trials_by_label,
    sample_slice=slice(50, 100),
    amplitude_func="mean",
):
    """
    Kaspar-style variance decomposition on one scalar amplitude per trial.

    Parameters
    ----------
    image_trials_by_label : dict[str, np.ndarray]
        Maps image label -> array of shape (n_trials, n_time) for one synapse.
    sample_slice : slice
        Samples within each peri-stimulus trace to use.
    amplitude_func : str
        How to collapse time within each trial.
        Options:
            "mean"   -> mean over samples in sample_slice
            "max"    -> max over samples
            "sum"    -> sum over samples
            "top10"  -> mean of top 10 samples within window

    Returns
    -------
    dict or None
    """
    labels = []
    amplitudes = []

    for label, arr in image_trials_by_label.items():
        arr = np.asarray(arr, dtype=float)
        if arr.ndim != 2 or arr.shape[0] == 0:
            continue

        arr = arr[:, sample_slice]
        if arr.size == 0:
            continue

        if amplitude_func == "mean":
            amp = np.nanmean(arr, axis=1)
        elif amplitude_func == "max":
            amp = np.nanmax(arr, axis=1)
        elif amplitude_func == "sum":
            amp = np.nansum(arr, axis=1)
        elif amplitude_func == "top10":
            k = min(10, arr.shape[1])
            part = np.partition(arr, -k, axis=1)[:, -k:]
            amp = np.nanmean(part, axis=1)
        else:
            raise ValueError(f"Unknown amplitude_func: {amplitude_func}")

        amplitudes.append(amp)
        labels.extend([label] * len(amp))

    if len(amplitudes) == 0:
        return None

    y = np.concatenate(amplitudes, axis=0)   # one scalar per presentation
    labels = np.asarray(labels)

    valid = np.isfinite(y)
    y = y[valid]
    labels = labels[valid]

    if y.size < 2:
        return None

    total_var = np.var(y)
    if not np.isfinite(total_var) or total_var <= 0:
        return None

    # 1) overall mean
    overall_mean = np.mean(y)
    resid_after_mean = y - overall_mean
    var_after_overall_mean = np.var(resid_after_mean)
    mean_fve = 1.0 - (var_after_overall_mean / total_var)

    # 2) subtract image-specific mean residuals, only within each image
    resid_after_image_means = np.empty_like(resid_after_mean)
    image_fve_by_label = {}

    for label in np.unique(labels):
        mask = labels == label
        resid_img = resid_after_mean[mask]

        image_mean_residual = np.mean(resid_img)
        final_resid_img = resid_img - image_mean_residual
        resid_after_image_means[mask] = final_resid_img

        var_after_mean_img = np.var(resid_img)
        var_after_image_img = np.var(final_resid_img)

        image_fve_by_label[label] = (
            var_after_mean_img - var_after_image_img
        ) / total_var

    var_after_image_means = np.var(resid_after_image_means)

    noise_fraction = var_after_image_means / total_var
    image_fve_total = (
        var_after_overall_mean - var_after_image_means
    ) / total_var

    return {
        "total_var": float(total_var),
        "var_after_overall_mean": float(var_after_overall_mean),
        "var_after_image_means": float(var_after_image_means),
        "mean_fve": float(mean_fve),
        "image_fve_total": float(image_fve_total),
        "noise_fraction": float(noise_fraction),
        "image_fve_by_label": image_fve_by_label,
        "n_trials_total": int(y.size),
        "n_images_present": int(len(np.unique(labels))),
        "overall_mean_amplitude": float(overall_mean),
    }

def compute_variance_decomposition_across_sessions(
    single_trial_paths,
    assets,
    act_summary,
    sample_slice=slice(50, 100),
    allowed_depths=(25, 100, 200, 250),
):
    """
    Compute full variance decomposition across sessions for activated synapses.

    Parameters
    ----------
    single_trial_paths : list[str or Path]
        Paths to glutamate_single_trial_df.npz, aligned with `assets`.
    assets : list
        Asset-like objects aligned to single_trial_paths.
        Required:
            asset.session_id
            asset.metadata['dmd1_depth']
            asset.metadata['dmd2_depth']
            asset.metadata['session_type']
            asset.metadata['session_#']  (or similar)
    act_summary : pd.DataFrame
        Concatenated activation summary table.
        Must contain:
            session_id, dmd, synapse_id, stimulus_family, response_class
    sample_slice : slice
        Peri-stimulus response window. Default uses samples 50:100.
    allowed_depths : iterable
        Canonical depth bins to prepare dictionaries for.

    Returns
    -------
    synapse_metrics_wide : pd.DataFrame
    synapse_metrics_long : pd.DataFrame
    grouped : dict
        Nested arrays for quick plotting by depth/session_type
    """
    # ------------------------------------------------------------
    # Activated synapse lookup
    # ------------------------------------------------------------
    activated_lookup = (
        act_summary.loc[
            (act_summary["stimulus_family"] == "image")
            & (act_summary["response_class"] == "activated"),
            ["session_id", "dmd", "synapse_id"],
        ]
        .drop_duplicates()
    )

    activated_sets = {
        (sid, dmd): set(df["synapse_id"].tolist())
        for (sid, dmd), df in activated_lookup.groupby(["session_id", "dmd"])
    }

    # ------------------------------------------------------------
    # Discover canonical image labels across sessions
    # ------------------------------------------------------------
    all_image_labels = set()
    for path in single_trial_paths:
        data = np.load(path, allow_pickle=True)["data"][0]
        for dmd_name in ["DMD1", "DMD2"]:
            if dmd_name not in data:
                continue
            if "image_identity" not in data[dmd_name]:
                continue
            all_image_labels.update(
                _image_label_from_key(k)
                for k in data[dmd_name]["image_identity"].keys()
            )

    all_image_labels = sorted(all_image_labels)

    # ------------------------------------------------------------
    # Main output rows
    # ------------------------------------------------------------
    wide_rows = []
    long_rows = []

    # ------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------
    for i, path in enumerate(single_trial_paths):
        asset = assets[i]
        session_id = asset.session_id
        metadata = getattr(asset, "metadata", {}) or {}

        print(f"Processing {session_id}")

        data = np.load(path, allow_pickle=True)["data"][0]

        depth_map = {
            "DMD1": metadata.get("dmd1_depth", np.nan),
            "DMD2": metadata.get("dmd2_depth", np.nan),
        }

        session_type = metadata.get("session_type", np.nan)
        session_order = _safe_session_order(metadata)

        for dmd_name in ["DMD1", "DMD2"]:
            if dmd_name not in data:
                continue
            if "image_identity" not in data[dmd_name]:
                continue

            activated_synapses = activated_sets.get((session_id, dmd_name), set())
            if len(activated_synapses) == 0:
                continue

            depth_um = depth_map[dmd_name]
            dmd_block = data[dmd_name]

            image_keys = list(dmd_block["image_identity"].keys())
            if len(image_keys) == 0:
                continue

            # Determine synapse ids
            if "synapse_ids" in dmd_block:
                synapse_ids = np.asarray(dmd_block["synapse_ids"])
            else:
                first_im = np.asarray(dmd_block["image_identity"][image_keys[0]])
                synapse_ids = np.arange(first_im.shape[1])

            # Build image arrays once
            image_arrays = {
                _image_label_from_key(k): np.asarray(
                    dmd_block["image_identity"][k], dtype=float
                )
                for k in image_keys
            }

            n_synapses = next(iter(image_arrays.values())).shape[1]

            for syn_idx in range(n_synapses):
                synapse_id = synapse_ids[syn_idx]

                if synapse_id not in activated_synapses:
                    continue

                # Collect single-trial responses per image for this synapse
                image_trials_by_label = {}
                for label, arr in image_arrays.items():
                    # arr shape: (n_trials, n_synapses, n_time)
                    if syn_idx >= arr.shape[1]:
                        continue
                    image_trials_by_label[label] = arr[:, syn_idx, :]

                metrics = _compute_variance_decomposition_for_synapse_amplitude(
                    image_trials_by_label=image_trials_by_label,
                    sample_slice=sample_slice,
                )

                if metrics is None:
                    continue

                # ----------------------------
                # Wide row: one row per synapse
                # ----------------------------
                row = {
                    "session_id": session_id,
                    "subject_id": getattr(asset, "subject_id", np.nan),
                    "dmd": dmd_name,
                    "depth_um": depth_um,
                    "session_type": session_type,
                    "session_order": session_order,
                    "synapse_id": synapse_id,
                    "synapse_index": syn_idx,
                    "n_trials_total": metrics["n_trials_total"],
                    "n_images_present": metrics["n_images_present"],
                    "total_var": metrics["total_var"],
                    "var_after_overall_mean": metrics["var_after_overall_mean"],
                    "var_after_image_means": metrics["var_after_image_means"],
                    "mean_fve": metrics["mean_fve"],
                    "image_fve_total": metrics["image_fve_total"],
                    "noise_fraction": metrics["noise_fraction"],
                }

                for lab in all_image_labels:
                    row[f"image_fve_{lab}"] = metrics["image_fve_by_label"].get(lab, np.nan)

                wide_rows.append(row)

                # ----------------------------
                # Long rows: tidy format
                # ----------------------------
                long_rows.append(
                    {
                        "session_id": session_id,
                        "subject_id": getattr(asset, "subject_id", np.nan),
                        "dmd": dmd_name,
                        "depth_um": depth_um,
                        "session_type": session_type,
                        "session_order": session_order,
                        "synapse_id": synapse_id,
                        "component": "mean",
                        "stimulus_label": "overall_mean",
                        "fraction": metrics["mean_fve"],
                    }
                )

                long_rows.append(
                    {
                        "session_id": session_id,
                        "subject_id": getattr(asset, "subject_id", np.nan),
                        "dmd": dmd_name,
                        "depth_um": depth_um,
                        "session_type": session_type,
                        "session_order": session_order,
                        "synapse_id": synapse_id,
                        "component": "image_total",
                        "stimulus_label": "all_images",
                        "fraction": metrics["image_fve_total"],
                    }
                )

                long_rows.append(
                    {
                        "session_id": session_id,
                        "subject_id": getattr(asset, "subject_id", np.nan),
                        "dmd": dmd_name,
                        "depth_um": depth_um,
                        "session_type": session_type,
                        "session_order": session_order,
                        "synapse_id": synapse_id,
                        "component": "noise",
                        "stimulus_label": "within_image_noise",
                        "fraction": metrics["noise_fraction"],
                    }
                )

                for lab in all_image_labels:
                    long_rows.append(
                        {
                            "session_id": session_id,
                            "subject_id": getattr(asset, "subject_id", np.nan),
                            "dmd": dmd_name,
                            "depth_um": depth_um,
                            "session_type": session_type,
                            "session_order": session_order,
                            "synapse_id": synapse_id,
                            "component": "image_by_identity",
                            "stimulus_label": lab,
                            "fraction": metrics["image_fve_by_label"].get(lab, np.nan),
                        }
                    )

    synapse_metrics_wide = pd.DataFrame(wide_rows)
    synapse_metrics_long = pd.DataFrame(long_rows)

    # ------------------------------------------------------------
    # Convenience grouped outputs for plotting
    # ------------------------------------------------------------
    grouped = {
        "mean_by_depth": {},
        "image_total_by_depth": {},
        "noise_by_depth": {},
        "mean_by_depth_session_type": {},
        "image_total_by_depth_session_type": {},
        "noise_by_depth_session_type": {},
    }

    for depth in allowed_depths:
        sub = synapse_metrics_wide.loc[synapse_metrics_wide["depth_um"] == depth]

        grouped["mean_by_depth"][depth] = sub["mean_fve"].dropna().to_numpy()
        grouped["image_total_by_depth"][depth] = sub["image_fve_total"].dropna().to_numpy()
        grouped["noise_by_depth"][depth] = sub["noise_fraction"].dropna().to_numpy()

        grouped["mean_by_depth_session_type"][depth] = {}
        grouped["image_total_by_depth_session_type"][depth] = {}
        grouped["noise_by_depth_session_type"][depth] = {}

        for sess_type, sub2 in sub.groupby("session_type", dropna=False):
            grouped["mean_by_depth_session_type"][depth][sess_type] = (
                sub2["mean_fve"].dropna().to_numpy()
            )
            grouped["image_total_by_depth_session_type"][depth][sess_type] = (
                sub2["image_fve_total"].dropna().to_numpy()
            )
            grouped["noise_by_depth_session_type"][depth][sess_type] = (
                sub2["noise_fraction"].dropna().to_numpy()
            )

    return synapse_metrics_wide, synapse_metrics_long, grouped

In [6]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

Loaded 56 session assets


In [7]:
st_paths = [glob.glob(os.path.join(asset.derived_dir,'**','glutamate_single_trial_df.npz'),recursive=True)[0] for asset in assets]

In [8]:
act_summary_path = r"C:\Users\andrew.shelton\Downloads\activation_summary.csv"
act_summary = pd.read_csv(act_summary_path)

In [ ]:
synapse_metrics_wide, synapse_metrics_long, grouped = (
    compute_variance_decomposition_across_sessions(
        single_trial_paths=st_paths,
        assets=assets,
        act_summary=act_summary,
        sample_slice=slice(50, 100),
    )
)

In [ ]:
plot_df = synapse_metrics_wide[["depth_um", "session_type", "mean_fve", "image_fve_total", "noise_fraction"]].copy()

In [ ]:
plot_df

In [ ]:
cl, cmap, cp = PNW_cmap.get_PNW_cmap('Sailboat', n_colors=4)
cp

In [ ]:
cp = cp[::-1]

In [ ]:
metric = 'image_fve_total'

fig,ax=plt.subplots(figsize=(4,4.5))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

for spine in ['left','right','top','bottom']:
    ax.spines[spine].set_linewidth(2)

sns.despine()
    
sns.stripplot(data = plot_df,x = 'depth_um', y = np.log10(plot_df[metric]),palette=cp,size=2)

means = []

for depth in plot_df['depth_um'].unique():
    dft = plot_df[plot_df['depth_um']==depth]
    mean = np.nanmean(np.log10(dft[metric]))
    means.append(mean)
ax.plot(means,color='k',zorder=11,lw=3,marker='o')
# ax.set_ylim(-0.01,0.075)

ax.set_ylabel('log(FVE)')
ax.set_xlabel('Depth from pia (\u03BCm)')

ax.set_title('Fraction of response variance\nexplained by Images')

fig.tight_layout()
filen = 'Image_FVE'
# save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
import itertools
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# ------------------------------------------------------------
# Pairwise depth comparisons
# ------------------------------------------------------------
depth_order = [25, 100, 200, 250]

pairwise_rows = []

for d1, d2 in itertools.combinations(depth_order, 2):
    x = plot_df.loc[plot_df["depth_um"] == d1, "image_fve_total"].dropna().to_numpy()
    y = plot_df.loc[plot_df["depth_um"] == d2, "image_fve_total"].dropna().to_numpy()

    if len(x) == 0 or len(y) == 0:
        continue

    stat, p = mannwhitneyu(x, y, alternative="two-sided")

    pairwise_rows.append(
        {
            "depth_1": d1,
            "depth_2": d2,
            "n_1": len(x),
            "n_2": len(y),
            "median_1": np.median(x),
            "median_2": np.median(y),
            "mean_1": np.mean(x),
            "mean_2": np.mean(y),
            "mw_u": stat,
            "p_uncorrected": p,
        }
    )

pairwise_stats = pd.DataFrame(pairwise_rows)

# ------------------------------------------------------------
# Multiple-comparisons correction
# ------------------------------------------------------------
reject, p_fdr, _, _ = multipletests(
    pairwise_stats["p_uncorrected"].values,
    alpha=0.05,
    method="fdr_bh",
)

pairwise_stats["p_fdr_bh"] = p_fdr
pairwise_stats["significant_fdr_bh"] = reject

pairwise_stats = pairwise_stats.sort_values("p_fdr_bh").reset_index(drop=True)

pairwise_stats